# Demo: Model Inference & Predictions

This notebook demonstrates how to load a trained model and run inference on new ECG data.

**Prerequisites**: Run `python scripts/train_pipeline.py` first to generate trained models.

## 1. Setup and Load Model

In [ ]:
import sys
from pathlib import Path
import json

# Add src directory to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.joblib import dump, load

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully!")

In [ ]:
# Import modules
from data_loader import load_data
from preprocessing import preprocess_data, encode_target
from models import load_model

print("Modules imported successfully!")

In [ ]:
# Load data and trained models
data_dir = Path.cwd().parent / 'data'
models_dir = Path.cwd().parent / 'models'
results_dir = Path.cwd().parent / 'results'

# Find dataset
data_file = data_dir / 'dataset_5_arrhythmia.arff'
if not data_file.exists():
    data_file = Path.cwd().parent.parent / 'dataset_5_arrhythmia.arff'

# Load full dataset
X_train, X_val, X_test, y_train, y_val, y_test, df_clean = load_data(
    str(data_file),
    target_col='Class',
    missing_value_strategy='drop',
    random_state=42
)

print(f" Dataset loaded: {X_test.shape[0]} test samples, {X_test.shape[1]} features")

# Preprocess test data
X_test_prep, _, _, preprocessing_info = preprocess_data(
    X_test, X_test, X_test,
    scale=True,
    encode_categorical=True
)

# Fix: properly preprocess with fitted scaler
X_train_prep, X_val_prep, X_test_prep, preprocessing_info = preprocess_data(
    X_train, X_val, X_test,
    scale=True,
    encode_categorical=True
)

# Encode target
y_train_enc, y_val_enc, y_test_enc, label_encoder = encode_target(y_train, y_val, y_test)

print(f" Data preprocessed: {X_test_prep.shape}")

In [ ]:
# Load best model
# Determine best model from results
metrics_file = results_dir / 'metrics.json'
best_model_name = 'random_forest'  # default

if metrics_file.exists():
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    
    # Find best model by macro F1
    best_f1 = 0
    for model_name, scores in metrics.items():
        if model_name != 'hyperparameters' and isinstance(scores, dict):
            f1 = scores.get('f1_macro', 0)
            if f1 > best_f1:
                best_f1 = f1
                best_model_name = model_name

model_path = models_dir / f"{best_model_name}.pkl"
if model_path.exists():
    best_model = load_model(str(model_path))
    print(f" Loaded best model: {best_model_name.replace('_', ' ').upper()}")
    print(f"   Model type: {type(best_model).__name__}")
else:
    print(f" Model file not found: {model_path}")
    print(f"   Available models: {list(models_dir.glob('*.pkl'))}")
    best_model = None

## 2. Sample Predictions

In [ ]:
if best_model is not None:
    # Select random test samples
    np.random.seed(42)
    sample_indices = np.random.choice(len(X_test_prep), size=min(5, len(X_test_prep)), replace=False)
    
    print("="*100)
    print("SAMPLE PREDICTIONS")
    print("="*100)
    
    for idx, sample_idx in enumerate(sample_indices, 1):
        # Get sample
        X_sample = X_test_prep[sample_idx:sample_idx+1]
        y_true = y_test_enc[sample_idx]
        
        # Predict
        y_pred = best_model.predict(X_sample)[0]
        
        # Get probabilities if available
        try:
            y_proba = best_model.predict_proba(X_sample)[0]
            confidence = np.max(y_proba)
        except:
            confidence = "N/A"
        
        # Decode labels
        true_label = label_encoder.inverse_transform([y_true])[0]
        pred_label = label_encoder.inverse_transform([y_pred])[0]
        
        # Check correctness
        correct = "CORRECT" if y_true == y_pred else " INCORRECT"
        
        print(f"\nSample {idx} (Test Index: {sample_idx})")
        print(f"  True Class:      {true_label}")
        print(f"  Predicted Class: {pred_label}")
        print(f"  Confidence:      {f'{confidence:.4f}' if confidence != 'N/A' else confidence}")
        print(f"  Status:          {correct}")

## 3. Batch Predictions on Test Set

In [ ]:
if best_model is not None:
    # Predict on entire test set
    y_pred_all = best_model.predict(X_test_prep)
    
    # Compute accuracy
    accuracy = np.mean(y_pred_all == y_test_enc)
    
    print("\nTest Set Performance:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Correct predictions: {np.sum(y_pred_all == y_test_enc)} / {len(y_test_enc)}")

## 4. Feature Importance (if applicable)

In [ ]:
if best_model is not None:
    # Check if model has feature importance
    if hasattr(best_model, 'feature_importances_'):
        importances = best_model.feature_importances_
        feature_names = X_test_prep.columns if hasattr(X_test_prep, 'columns') else [f'Feature {i}' for i in range(len(importances))]
        
        # Get top 10 features
        top_indices = np.argsort(importances)[-10:][::-1]
        top_features = [feature_names[i] for i in top_indices]
        top_importances = importances[top_indices]
        
        print("\nTop 10 Most Important Features:")
        print("="*50)
        for rank, (feature, imp) in enumerate(zip(top_features, top_importances), 1):
            print(f"{rank:2d}. {str(feature):30s} {imp:.6f}")
        
        # Visualization
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(top_features, top_importances, color='steelblue')
        ax.set_xlabel('Importance Score', fontsize=12, fontweight='bold')
        ax.set_title(f'Top 10 Feature Importance - {best_model_name.replace("_", " ").title()}', fontsize=14, fontweight='bold')
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()
    
    elif hasattr(best_model, 'coef_'):
        print("\n Model uses coefficients instead of feature importance")
        print("   (Logistic Regression - weights are available but interpretation is complex for multi-class)")
    else:
        print("\n⚠ This model type doesn't provide feature importance information")

## 5. Confusion Analysis

In [ ]:
if best_model is not None:
    from sklearn.metrics import confusion_matrix, classification_report
    
    # Compute confusion matrix
    cm = confusion_matrix(y_test_enc, y_pred_all)
    
    # Per-class metrics
    print("\nPer-Class Metrics:")
    print("="*80)
    
    report = classification_report(
        y_test_enc, y_pred_all,
        target_names=label_encoder.classes_,
        digits=4
    )
    print(report)
    
    # Find hardest classes (lowest F1)
    print("\nClasses with Lowest F1 Scores (Most Difficult to Predict):")
    print("="*80)
    lines = report.split('\n')
    # Extract F1 scores
    f1_scores = []
    for i, line in enumerate(lines[2:-3]):
        if len(line.strip()) > 0:
            parts = line.split()
            if len(parts) >= 4:
                try:
                    f1 = float(parts[-1])
                    class_name = ' '.join(parts[:-3])
                    f1_scores.append((class_name, f1))
                except:
                    pass
    
    if f1_scores:
        f1_scores_sorted = sorted(f1_scores, key=lambda x: x[1])
        for class_name, f1 in f1_scores_sorted[:5]:
            print(f"  {class_name:20s} F1: {f1:.4f}")

## 6. Usage Instructions for New Data

In [ ]:
print("="*80)
print("HOW TO USE THE TRAINED MODEL ON NEW DATA")
print("="*80)

print("""
1. PREPARE YOUR DATA
   - Load ECG data in ARFF format or convert to DataFrame
   - Ensure same features as training set (279 features)
   - Handle missing values (drop rows with NaN)

2. PREPROCESS THE DATA
   - Encode categorical features with LabelEncoder
   - Scale numeric features with the fitted StandardScaler
   - IMPORTANT: Use the same scaler from training to ensure consistency

3. MAKE PREDICTIONS
   - Load the trained model: model = load_model('models/best_model.pkl')
   - Predict: y_pred = model.predict(X_new_preprocessed)
   - Get probabilities: y_proba = model.predict_proba(X_new_preprocessed)

4. INTERPRET RESULTS
   - y_pred: Class labels (0-15, map back to original class names)
   - y_proba: Confidence scores for each class
   - Select class with highest probability for most confident prediction

EXAMPLE CODE:
```python
from src.models import load_model
from src.preprocessing import preprocess_data
import pickle

# Load fitted scaler and encoders from training
with open('preprocessing_info.pkl', 'rb') as f:
    preprocessing_info = pickle.load(f)

# Load model
model = load_model('models/random_forest.pkl')

# Preprocess new data
# (encode categoricals, scale numerics using fitted scaler)
X_new_preprocessed = preprocess_with_fitted_transformers(X_new, preprocessing_info)

# Predict
y_pred = model.predict(X_new_preprocessed)
y_proba = model.predict_proba(X_new_preprocessed)

# Get class labels
pred_classes = label_encoder.inverse_transform(y_pred)
```

""")